**STEP 1: Simulation**

In [1]:
import numpy as np
import random
from scipy.stats import wishart
def simulate_matrix(n, d, no_components,rho,abs_bound_X):
    ###Form the X matrix
    X = np.zeros((0, d))
    # Use base_value and reminder to evenly distributed the number of samples in each component
    def generate_components(n, no_components):
        base_value = n // no_components
        remainder = n % no_components
        components = [base_value] * no_components
        for i in range(remainder):
            components[i] += 1
        random.shuffle(components)
        if sum(components) > n:
            components[n-1] -= sum(components) - n
        elif sum(components) < n:
            components[n-1] += n - sum(components)
        return components
    # Generate these multivariate gaussian components in different levels of mean and cov.
    num_list = generate_components(n, no_components)
    for i in range(no_components):
        # each level's mean value, grow by 4's power.
        mean_value = 5**(i+1)
        # 0.8 probability of 1 and 0.2 probability of 0 for creating a sparse mean vector.
        mean_vector = np.random.binomial(1, 0.8, size = d)
        mean = mean_value * mean_vector
        # randomly generate a cov matrix and make sure that it is symmetric.
        cov = wishart.rvs(df=d, scale=np.eye(d))
        # add a small value to the diagonal to ensure positive-definiteness
        cov += np.eye(d) * 1e-6
        # generate a component of X.
        X_ = np.random.multivariate_normal(mean, cov, num_list[i])
        X_ = np.array(X_)
        # append the component to the list.
        X = np.vstack((X, X_))
    X = np.array(X)
    # Set the elements < 3 to 0 to make it a sparse matrix
    X[abs(X) < abs_bound_X] = 0
    ###Form the fixed beta
    np.random.seed(39)
    beta_value = np.random.binomial(1, 0.8, size = d)
    beta = (beta_value * np.random.uniform(-1, 1, size = d)).reshape(-1,1)
    beta = np.array(beta)
    np.random.seed(None)
    ###Form the error term - epsilon, which has the auto-correlation structure in time series analysis.
    r = rho # autocorrelation coefficient
    epsilon_matrix = np.zeros((0,1)) # define a zero initial matrix for vstack
    ###Form the error term in the same levels
    for i in range(no_components):
        epsilon = np.zeros((num_list[i],1))
        epsilon[0] = np.random.normal((5**(i+1))/100,1)
        for t in range(1,num_list[i]):
            epsilon[t] = r * epsilon[t-1] + np.random.normal((5**(i+1))/100,1)
        epsilon = np.array(epsilon)
        epsilon_matrix = np.vstack((epsilon_matrix, epsilon))
    epsilon_matrix = np.array(epsilon_matrix)
    epsilon = epsilon_matrix
    ###Form the Y matrix
    Y = np.array(X @ beta + epsilon)
    ###Count the number of elements < 0.1 in X and Y
    return X,Y,beta,epsilon

# Matrix Settled
n = 10**4       # number of rows         
d = 500        # number of columns
no_components = 4 # number of levels of multivariate gaussian distribution in X
rho = 0.7 # autocorrelation coefficient in time series analysis
abs_bound_X = 2 # absolute bound for X to set elements to be 0
X,Y,beta,epsilon = simulate_matrix(n, d, no_components,rho,abs_bound_X)
print("The simulated matrix X is:")
print(X)
print("The simulated vector Y is:")
print(Y)

The simulated matrix X is:
[[ -9.93109637 -15.87857894  56.15102235 ...   0.          52.59960313
  -22.02577031]
 [-42.81475227  14.55911338  -2.63411147 ...  -2.03105876  -7.27393311
    3.99885107]
 [ -3.43117167  -9.57468905  39.06102919 ...  21.0156462   38.07171818
    8.20386306]
 ...
 [623.42385576 655.04466633 613.28301795 ... 662.10773089 625.41359862
  652.3630372 ]
 [597.58335661 612.09300035 675.43047014 ... 650.90544751 631.38218328
  580.90238799]
 [616.57884438 677.78233409 577.05561381 ... 615.14702295 611.87252707
  629.76875061]]
The simulated vector Y is:
[[-378.22952024]
 [ 204.27329933]
 [ 589.73439792]
 ...
 [2841.31923714]
 [2589.88478031]
 [2731.0673479 ]]


*Define functions for calculating ratio and norms*

In [2]:
def calculate_the_norm_square(A,b,x_selected):
    return (np.linalg.norm(A @ x_selected - b)) ** 2
def abs_error_ratio(A,b,x_hat_bar,x_star):
    return calculate_the_norm_square(A,b,x_hat_bar) - calculate_the_norm_square(A,b,x_star) / calculate_the_norm_square(A,b,x_star)

*Try to figure out what x_star should be exiquisitely*

In [3]:
from scipy.linalg import cho_factor, cho_solve
import time
def solver(A,b,solver):
    # cholesky decomposition algorithm
    if solver == 'cholesky':
        L,low = cho_factor(A.T@A)
        x_star = cho_solve((L,low),A.T@b)
    # Direct solver
    elif solver == 'direct':
        x_star = np.linalg.inv(A.T@A)@A.T@b
    return x_star
start_cholesky = time.time()
x_star_cholesky = solver(X,Y,'cholesky')
end_cholesky = time.time()
execution_cholesky = end_cholesky - start_cholesky
start_direct = time.time()
x_star_direct = solver(X,Y,'direct')
end_direct = time.time()
execution_direct = end_direct - start_direct
print("The execution time of cholesky decomposition is:")
print(execution_cholesky)
print("The execution time of direct solver is:")
print(execution_direct)
norm_cholesky = calculate_the_norm_square(X,Y,x_star_cholesky)
norm_direct = calculate_the_norm_square(X,Y,x_star_direct)
print("The norm square of x_star calculated as by cholesky and direct solver respectively are:")
print(norm_cholesky, norm_direct)
minimum_norm_square = min(norm_cholesky, norm_direct)
print("Here we output the minimum of the norm suqare for x_star calculated as:")
print(minimum_norm_square)
print("Their difference by cholesky minus direct solver is:")
print(norm_cholesky - norm_direct)

The execution time of cholesky decomposition is:
0.019597768783569336
The execution time of direct solver is:
0.13758277893066406
The norm square of x_star calculated as by cholesky and direct solver respectively are:
18885.544690663548 18885.54469066373
Here we output the minimum of the norm suqare for x_star calculated as:
18885.544690663548
Their difference by cholesky minus direct solver is:
-1.8189894035458565e-10


###It is obvious that norm difference of cholsky and direct solver is very small.

**Consider the number of rows (n) of the matrix X and Y in the scale of powers of 10.**<br>
*In my computation environment, when n = 10^6, the computation time of simulation is only 15.2s in one test case.*<br>
*X_star by cholesky takes 0.939s and x_star by direct solver takes 2.464s*<br>
*But when n = 10^7, the computation time of simulation is as large as 21m 9.5s in one test case.*<br>
*X_star by cholesky and x_star by direct solver even collapse in time and memory.*

In [4]:
###From the above different norm and optimal solver calculations, it is obvious that:
###the direct solver and cholesky decomposition are the optimal solvers.
###Since their norms are very very very nearly the same
###we choose the direct solver as our optimal solver when matrix is not so large.
###And we choose the cholesky decomposition as our optimal solver when matrix is so large.
x_Star = x_star_direct
norm_Star = norm_direct
print("The optimal solver is:")
print(x_Star)

The optimal solver is:
[[-1.16536120e-01]
 [ 7.47652574e-01]
 [ 8.47057333e-04]
 [-4.42597939e-01]
 [ 9.57032247e-01]
 [ 3.74452169e-01]
 [-1.35597849e-01]
 [-8.99590241e-01]
 [ 9.55457886e-02]
 [-1.07175222e-05]
 [-1.46075868e-03]
 [ 3.20547094e-04]
 [-3.10806220e-04]
 [-1.69443803e-01]
 [-3.30792112e-04]
 [ 6.16592655e-04]
 [-5.98237420e-01]
 [ 8.65114402e-01]
 [-4.58372160e-01]
 [ 8.92129107e-04]
 [ 4.11830836e-01]
 [-9.63173254e-04]
 [-4.84827466e-01]
 [-4.05943421e-01]
 [ 9.08059183e-02]
 [ 3.30975636e-04]
 [-1.72825585e-01]
 [ 1.07446520e-01]
 [-2.58370627e-01]
 [-3.17913083e-01]
 [-1.16622172e-01]
 [ 1.38087058e-01]
 [-9.54385462e-02]
 [ 1.51664277e-04]
 [ 5.99439392e-01]
 [ 4.96244713e-01]
 [ 9.62088491e-01]
 [-5.93786804e-04]
 [-6.09597966e-01]
 [ 6.97369722e-04]
 [-6.21304575e-01]
 [ 3.95555732e-01]
 [ 2.07189075e-01]
 [-4.19951786e-04]
 [-6.27010319e-01]
 [-1.15660610e-01]
 [ 2.15268123e-01]
 [-6.76718172e-01]
 [-2.02140289e-01]
 [-9.22977204e-01]
 [ 8.34949400e-01]
 [ 8.833

**STEP 2: Uniform Sampling sketch // Algorithm 1: Distributed Randomized Regression**

In [5]:
# e.g. our desired sketching size is m = 1000
from operator import index


m = 1000

# Algorithm 1 inserting inside uniform sampling: Distributed Randomized Regression

# S_k @ A here is just computed as A [uniformly_sampled_index]
# As S_k here is just a diagnoal matrix of 1 or 0 where sampled rows have 1 as value
def uniform_sampling(A,b,n,m,q):
    x_hat_list = []
    for i in range(q):
        index = np.random.choice(n, size=m, replace=False)
        A_sk = A[index]
        b_sk = b[index]
        x_hat = np.linalg.inv(A_sk.T @ A_sk) @ A_sk.T @ b_sk
        x_hat_list.append(x_hat)
    x_bar = sum(x_hat_list) / q
    return x_bar
